# DarkIR: Robust Low-Light Image Restoration (CVPR 2025)

This notebook runs **DarkIR** inference and evaluation on Kaggle.

**What this notebook covers:**
1. Install dependencies and clone the repo
2. Download pretrained weights from HuggingFace
3. Load the model (no DDP — single GPU/CPU)
4. Run inference on your own images
5. (Optional) Evaluate with PSNR / SSIM / LPIPS on a paired dataset

**Accelerator:** Set *Settings → Accelerator → GPU T4 x2* (or GPU P100) for best speed.

## 1. Install dependencies

In [ ]:
# Packages not pre-installed on Kaggle
!pip install -q ptflops lpips pytorch-msssim pyiqa einops

## 2. Clone the DarkIR repository

In [ ]:
import os

REPO_DIR = '/kaggle/working/DarkIR'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Fundacion-Cidaut/DarkIR.git {REPO_DIR}
else:
    print('Repo already cloned.')

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 3. Download pretrained weights

Weights are hosted on HuggingFace at [`Cidaut/DarkIR`](https://huggingface.co/Cidaut/DarkIR).

Two variants are available:
| Variant | Width | Params | GMACs |
|---|---|---|---|
| `DarkIR_32width.pt` | 32 | 3.31 M | 7.25 |
| `DarkIR_64width.pt` | 64 | 12.96 M | 27.19 |

Change `MODEL_VARIANT` below to switch.

In [ ]:
from huggingface_hub import hf_hub_download

# Choose 'DarkIR_32width.pt' for the lighter model or 'DarkIR_64width.pt' for the full model
MODEL_VARIANT = 'DarkIR_64width.pt'

weights_path = hf_hub_download(
    repo_id='Cidaut/DarkIR',
    filename=MODEL_VARIANT,
)
print('Weights downloaded to:', weights_path)

## 4. Imports and configuration

The original code uses YAML config files + argparse. Here we inline the config as a plain dict so everything runs in the notebook without any CLI arguments.

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.transforms import Resize
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
from pathlib import Path

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('Using device:', device)

# ── Model configuration (mirrors options/inference/LOLBlur.yml) ──────────────
# Switch width to 32 if you downloaded DarkIR_32width.pt
MODEL_WIDTH = 64  # 32 for DarkIR-m, 64 for DarkIR-l

model_cfg = dict(
    name             = 'DarkIR',
    img_channels     = 3,
    width            = MODEL_WIDTH,
    middle_blk_num_enc = 2,
    middle_blk_num_dec = 2,
    enc_blk_nums     = [1, 2, 3],
    dec_blk_nums     = [3, 1, 1],
    dilations        = [1, 4, 9],
    extra_depth_wise = True,
)

## 5. Build and load the model

The training code wraps the model in `DistributedDataParallel` (DDP), which adds a `module.` prefix to every weight key. On Kaggle we run on a single device, so we load the raw weights directly — no DDP needed.

In [ ]:
from archs.DarkIR import DarkIR

def build_model(cfg, weights_path, device):
    model = DarkIR(
        img_channel       = cfg['img_channels'],
        width             = cfg['width'],
        middle_blk_num_enc= cfg['middle_blk_num_enc'],
        middle_blk_num_dec= cfg['middle_blk_num_dec'],
        enc_blk_nums      = cfg['enc_blk_nums'],
        dec_blk_nums      = cfg['dec_blk_nums'],
        dilations         = cfg['dilations'],
        extra_depth_wise  = cfg['extra_depth_wise'],
    )

    checkpoint = torch.load(weights_path, map_location='cpu', weights_only=False)

    # Checkpoints store weights under the 'params' key (without the DDP 'module.' prefix)
    state_dict = checkpoint['params']
    model.load_state_dict(state_dict)
    print('Weights loaded successfully.')

    model.eval()
    model.to(device)
    return model


model = build_model(model_cfg, weights_path, device)

## 6. Inference utilities

In [ ]:
pil_to_tensor  = transforms.ToTensor()
tensor_to_pil  = transforms.ToPILImage()


def load_image(path):
    """Load an image from disk and return a (1, 3, H, W) float tensor in [0, 1]."""
    img = Image.open(path).convert('RGB')
    return pil_to_tensor(img).unsqueeze(0)


def pad_to_multiple(tensor, multiple=8):
    """Pad spatial dims so both are divisible by `multiple`."""
    _, _, H, W = tensor.shape
    pad_h = (multiple - H % multiple) % multiple
    pad_w = (multiple - W % multiple) % multiple
    return F.pad(tensor, (0, pad_w, 0, pad_h), value=0)


@torch.no_grad()
def enhance_image(model, tensor, device, max_side=1500):
    """
    Run DarkIR on a single image tensor.

    Very large images are downsampled before inference and upsampled back
    afterward to keep memory usage reasonable on Kaggle GPUs.
    """
    tensor = tensor.to(device)
    _, _, H, W = tensor.shape

    # Optional downscale for very large images
    if H >= max_side or W >= max_side:
        new_h, new_w = H // 2, W // 2
        tensor = Resize((new_h, new_w))(tensor)
        resized = True
    else:
        resized = False

    padded = pad_to_multiple(tensor)
    output = model(padded, side_loss=False)

    # Crop padding and restore original size
    _, _, ph, pw = padded.shape
    out_h = H // 2 if resized else H
    out_w = W // 2 if resized else W
    output = output[:, :, :out_h, :out_w]

    if resized:
        output = Resize((H, W))(output)

    output = torch.clamp(output, 0.0, 1.0)
    return output.cpu()


def tensor_to_numpy(tensor):
    """Convert a (1, 3, H, W) or (3, H, W) tensor to a uint8 HxWx3 numpy array."""
    t = tensor.squeeze(0)
    return (t.permute(1, 2, 0).numpy() * 255).clip(0, 255).astype(np.uint8)


def show_pair(low_tensor, enhanced_tensor, title_low='Input (low-light)', title_out='DarkIR output'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(tensor_to_numpy(low_tensor))
    axes[0].set_title(title_low, fontsize=13)
    axes[0].axis('off')
    axes[1].imshow(tensor_to_numpy(enhanced_tensor))
    axes[1].set_title(title_out, fontsize=13)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()


print('Inference utilities ready.')

## 7. Single-image inference

Point `IMAGE_PATH` at any low-light image — for example one of the sample images bundled with the repo, or your own upload.

In [ ]:
# ── Change this to your image path ───────────────────────────────────────────
IMAGE_PATH = f'{REPO_DIR}/assets/teaser/0085_low.png'
# ─────────────────────────────────────────────────────────────────────────────

input_tensor    = load_image(IMAGE_PATH)
enhanced_tensor = enhance_image(model, input_tensor, device)

show_pair(input_tensor, enhanced_tensor)

### Save the result to disk

In [ ]:
output_path = '/kaggle/working/enhanced_result.png'
tensor_to_pil(enhanced_tensor.squeeze(0)).save(output_path)
print('Saved to', output_path)

## 8. Batch inference on a folder

Set `INPUT_DIR` to the folder containing your low-light images. Results are saved to `OUTPUT_DIR`.

In [ ]:
INPUT_DIR  = f'{REPO_DIR}/assets/qualis/inputs'
OUTPUT_DIR = '/kaggle/working/batch_results'

os.makedirs(OUTPUT_DIR, exist_ok=True)

SUPPORTED = {'.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG'}
image_paths = [p for p in Path(INPUT_DIR).iterdir() if p.suffix in SUPPORTED]

print(f'Found {len(image_paths)} images in {INPUT_DIR}')

for img_path in tqdm(image_paths, desc='Enhancing'):
    inp = load_image(img_path)
    out = enhance_image(model, inp, device)
    save_path = os.path.join(OUTPUT_DIR, img_path.name)
    tensor_to_pil(out.squeeze(0)).save(save_path)

print('Done. Results saved to', OUTPUT_DIR)

### Quick visual check on the batch results

In [ ]:
result_files = sorted(Path(OUTPUT_DIR).iterdir())[:4]  # show up to 4

fig, axes = plt.subplots(2, len(result_files), figsize=(5 * len(result_files), 8))

for col, res_path in enumerate(result_files):
    src_path = Path(INPUT_DIR) / res_path.name
    axes[0, col].imshow(np.array(Image.open(src_path).convert('RGB')))
    axes[0, col].set_title('Input', fontsize=10)
    axes[0, col].axis('off')

    axes[1, col].imshow(np.array(Image.open(res_path).convert('RGB')))
    axes[1, col].set_title('DarkIR', fontsize=10)
    axes[1, col].axis('off')

plt.suptitle('Batch results', fontsize=13)
plt.tight_layout()
plt.show()

## 9. (Optional) Quantitative evaluation on a paired dataset

This section computes **PSNR**, **SSIM**, and **LPIPS** when you have matching low/high-light image pairs.

Set the two directories below to your dataset's input (low-light) and ground-truth (normal-light) folders. The filenames must match between the two folders.

> **Tip:** Upload a benchmark dataset (e.g. LOLv2, LOL-Blur test split) via *Add Data* → *Upload*.

In [ ]:
# ── Set these to your dataset paths ──────────────────────────────────────────
LOW_DIR  = '/kaggle/input/your-dataset/low'   # low-light images
HIGH_DIR = '/kaggle/input/your-dataset/high'  # ground-truth images
# ─────────────────────────────────────────────────────────────────────────────

# Verify the directories exist before continuing
if not os.path.isdir(LOW_DIR) or not os.path.isdir(HIGH_DIR):
    print('Please set LOW_DIR and HIGH_DIR to valid paths. Skipping evaluation.')
else:
    print(f'Low:  {LOW_DIR}')
    print(f'High: {HIGH_DIR}')

In [ ]:
# Run this cell only if the directories above are set correctly

if os.path.isdir(LOW_DIR) and os.path.isdir(HIGH_DIR):
    from lpips import LPIPS
    from pytorch_msssim import ssim as calc_ssim_fn

    SUPPORTED = {'.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG'}
    low_paths = sorted([p for p in Path(LOW_DIR).iterdir()  if p.suffix in SUPPORTED])
    high_paths= sorted([p for p in Path(HIGH_DIR).iterdir() if p.suffix in SUPPORTED])

    assert len(low_paths) == len(high_paths), \
        f'Mismatch: {len(low_paths)} low vs {len(high_paths)} high images'
    print(f'Evaluating on {len(low_paths)} image pairs...')

    lpips_fn = LPIPS(net='vgg', verbose=False).to(device)

    psnr_list, ssim_list, lpips_list = [], [], []

    for low_p, high_p in tqdm(zip(low_paths, high_paths), total=len(low_paths), desc='Evaluating'):
        low_t  = load_image(low_p).to(device)
        high_t = load_image(high_p).to(device)

        with torch.no_grad():
            enhanced = enhance_image(model, low_t, device)
            enhanced = enhanced.to(device)

            # PSNR
            mse  = torch.mean((high_t - enhanced) ** 2).item()
            psnr = 20 * np.log10(1.0 / np.sqrt(mse + 1e-8))

            # SSIM  (pytorch-msssim expects values in [0, 1])
            ssim_val = calc_ssim_fn(enhanced, high_t, data_range=1.0, size_average=True).item()

            # LPIPS
            # LPIPS expects images in [-1, 1]
            lpips_val = lpips_fn(enhanced * 2 - 1, high_t * 2 - 1).item()

        psnr_list.append(psnr)
        ssim_list.append(ssim_val)
        lpips_list.append(lpips_val)

    print(f'\n Results over {len(low_paths)} pairs:')
    print(f'  PSNR  : {np.mean(psnr_list):.4f} dB')
    print(f'  SSIM  : {np.mean(ssim_list):.4f}')
    print(f'  LPIPS : {np.mean(lpips_list):.4f}')

## Notes

- **Weight key mismatch?** The training code wraps the model in `DistributedDataParallel`, which prefixes all keys with `module.`. The checkpoint's `params` dict stores keys *without* this prefix, so `build_model()` above loads them directly. If you see a `missing key` error mentioning `module.`, you loaded the wrong checkpoint format — check that `checkpoint['params']` exists.
- **Out of memory?** Lower `max_side` in `enhance_image()` or switch to the 32-width variant (`DarkIR_32width.pt` + `MODEL_WIDTH = 32`).
- **Gradio demo:** The repo ships an `app.py` that runs a Gradio interface. You can launch it in a notebook with `!python {REPO_DIR}/app.py` and use the Kaggle public URL to access it.